In [1]:
import pandas as pd

data = r"/home/aamer/smol_llm/data/stage00/cpt_corpus.jsonl"
df = pd.read_json(data, lines=True)

HELD_OUT = 200
SEED = 20260906  # same seed as build_dataset.py

held_out = df.sample(n=HELD_OUT, random_state=SEED)
train = df.drop(held_out.index)

print(f"total={len(df)}  train={len(train)}  held_out={len(held_out)}")
assert len(train) + len(held_out) == len(df)
assert set(train.index).isdisjoint(held_out.index)

train_path = r"/home/aamer/smol_llm/data/stage00/cpt_train.jsonl"
held_path = r"/home/aamer/smol_llm/data/stage00/cpt_heldout.jsonl"
train.to_json(train_path, orient="records", lines=True)
held_out.to_json(held_path, orient="records", lines=True)
print(f"wrote {train_path}")
print(f"wrote {held_path}")


total=9869  train=9669  held_out=200
wrote /home/aamer/smol_llm/data/stage00/cpt_train.jsonl
wrote /home/aamer/smol_llm/data/stage00/cpt_heldout.jsonl


In [2]:
# Testing model on the held-out set
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "HuggingFaceTB/SmolLM2-360M"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype = torch.float32)

print(f"Loaded model {model_id} with {model.num_parameters()/1e6:.2f}M parameters")
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Tokenizer EoS token: {tokenizer.eos_token}  id={tokenizer.eos_token_id}")


/home/aamer/smol_llm/.venv-sft/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1440.50it/s]


Loaded model HuggingFaceTB/SmolLM2-360M with 361.82M parameters
Tokenizer vocab size: 49152
Tokenizer EoS token: <|endoftext|>  id=0


# Testing Generation

In [3]:
# Testing generation 
import torch 

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

In [4]:
prompt = held_out.iloc[0]["text"][:400]
print(f"Prompt: {prompt}")

inputs = tokenizer(prompt, return_tensors="pt").to(device)
print(f"Input IDs: {inputs['input_ids']}, len={len(inputs['input_ids'][0])}")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.pad_token_id,
    )

generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(f"Generated text: {generated_text}")

Prompt: Effect of Eltrombopag Plus G-CSF on Human CD34+ Cell Mobilization in Multiple Myeloma Patients Undergoing ASCT

Eltrombopag may improve the cell collection available for Autologous Stem Cell Transplant(ASCT). The overall goal is to determine if adding Eltrombopag to the standard ASCT will increase the number of blood cells collected and reduce the number of times blood needs to be collected. This 
Input IDs: tensor([[22365,   282,   414,  2392, 22618,   371,   454,  9933,   452,    29,
          5826,    54,   335,  4996,  6559,    35,    36,    27, 11036, 26408,
         20768,   281, 16560,  3361,   299,  4826, 14672,  2995,  4304, 11131,
          5335,   198,   198,    53,  2392, 22618,   371,   454,   654,  1947,
           260,  1297,  3854,  1770,   327,  4490, 30014, 28065, 11036, 40415,
            24,  2211,  5335,   595,   378,  3043,  3491,   314,   288,  3346,
           585,  4990,   414,  2392, 22618,   371,   454,   288,   260,  2920,
         11131,  5335,   52

# Preparing the heldout set

In [5]:
# Test before CPT 

held_path = r"/home/aamer/smol_llm/data/stage00/cpt_heldout.jsonl"
df = pd.read_json(held_path, lines=True)

corpus = []
for i, row in df.iterrows():
    corpus.append(row["text"] + tokenizer.eos_token)

corpus = "".join(corpus)

# Tokenizing the corpus
inputs = tokenizer(corpus, return_tensors="pt") # Dont need truncation here since we will split into chunks later

print(inputs["input_ids"][0])

# Splitting the corpus into chunks of 1024 tokens
chunk_size = 1024
chunks = []
for i in range(0, inputs["input_ids"].shape[1], chunk_size):
    chunk = inputs["input_ids"][:, i:i + chunk_size]
    chunks.append(chunk)

print(f"Total chunks: {len(chunks)}")

# Drop the last chunk if it's smaller than chunk_size
if chunks[-1].shape[1] < chunk_size:
    chunks = chunks[:-1]

print(f"Total chunks after dropping last: {len(chunks)}")


Token indices sequence length is longer than the specified maximum sequence length for this model (112895 > 8192). Running this sequence through the model will result in indexing errors


tensor([22365,   282,   414,  ...,    40,    30,     0])
Total chunks: 111
Total chunks after dropping last: 110


In [6]:
# Measuring perplexity on the held-out set
import math

total_loss = 0.0
total_tokens = 0

model.eval()
for i, chunk in enumerate(chunks):
    with torch.no_grad():
        outputs = model(chunk.to(device), labels=chunk.to(device))
        loss = outputs.loss.item()
        total_loss += loss * chunk.shape[1]  # Multiply by number of tokens in the chunk
        total_tokens += chunk.shape[1]

total_characters = sum(len(row["text"]) for _, row in df.iterrows())
print(f"Total loss: {total_loss:.4f}")
print(f"Total tokens: {total_tokens}")
print(f"Total characters in held-out set: {total_characters}")

perplexity = math.exp(total_loss / total_tokens)
print(f"Perplexity on held-out set: {perplexity:.2f}")

model_agnostic_perplexity = math.exp(total_loss / total_characters)
print(f"Model-agnostic perplexity on held-out set: {model_agnostic_perplexity:.2f}")


Total loss: 264920.0659
Total tokens: 112640
Total characters in held-out set: 535430
Perplexity on held-out set: 10.51
Model-agnostic perplexity on held-out set: 1.64


In [7]:
# Measuring perplexity on general test set to test for forgetting
from datasets import load_dataset

wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")

lines = [t for t in wiki["text"]
         if t.strip() and not t.strip().startswith("=")]   # drop blanks + headings

general_corpus = "".join(lines)
print(len(general_corpus), "chars")


1264384 chars


In [8]:
# Tokenize the general test set and split into chunks of 1024 tokens
general_inputs = tokenizer(general_corpus, return_tensors="pt")
print(general_inputs["input_ids"][0])


general_chunks= []
chunk_size = 1024 

for i in range(0, len(general_inputs["input_ids"][0]), chunk_size):
    chunk = general_inputs["input_ids"][0][i:i + chunk_size]
    general_chunks.append(chunk)

if len(general_chunks[-1]) < chunk_size:
    general_chunks = general_chunks[:-1]

print(f"Total chunks in general test set: {len(general_chunks)}")


tensor([6356,  389, 9226,  ..., 2393, 1673, 3717])
Total chunks in general test set: 288


In [9]:
# Get the PPL on general set 
total_loss = 0.0
total_tokens = 0

model.eval()
for i, chunk in enumerate(general_chunks):
    with torch.no_grad():
        outputs = model(chunk.unsqueeze(0).to(device), labels=chunk.unsqueeze(0).to(device))
        loss = outputs.loss.item()
        total_loss += loss * chunk.shape[0]  # Multiply by number of tokens in the chunk
        total_tokens += chunk.shape[0]
ppl = torch.exp(torch.tensor(total_loss / total_tokens))
print(f"Perplexity on general test set: {ppl.item()}")


Perplexity on general test set: 13.118131637573242


# Qwen Tests

In [ ]:
# # Using a bigger model (Qwen/Qwen3-4B-Instruct-2507)
# qwen_model_id = "Qwen/Qwen3-4B-Instruct-2507"
# qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_id)
# qwen_model = AutoModelForCausalLM.from_pretrained(qwen_model_id)

# total_params = sum(p.numel() for p in qwen_model.parameters())
# print(f"Loaded model {qwen_model_id} with {total_params/1e6:.2f}M parameters")
# print(f"Tokenizer vocab size: {qwen_tokenizer.vocab_size}")
# print(f"Tokenizer EoS token: {qwen_tokenizer.eos_token}  id={qwen_tokenizer.eos_token_id}")


Loading weights: 100%|██████████| 398/398 [-00:00<00:00, -870.83it/s]

Loaded model Qwen/Qwen3-4B-Instruct-2507 with 4022.47M parameters
Tokenizer vocab size: 151643
Tokenizer EoS token: <|im_end|>  id=151645


In [ ]:
# corpus = []
# for i, row in df.iterrows():
#     corpus.append(row["text"] + qwen_tokenizer.eos_token)

# corpus = "".join(corpus)

# # Tokenizing the corpus
# inputs = qwen_tokenizer(corpus, return_tensors="pt") # Dont need truncation here since we will split into chunks later

# print(inputs["input_ids"][0])

# # Splitting the corpus into chunks of 1024 tokens
# chunk_size = 1024
# chunks = []
# for i in range(0, inputs["input_ids"].shape[1], chunk_size):
#     chunk = inputs["input_ids"][:, i:i + chunk_size]
#     chunks.append(chunk)

# print(f"Total chunks: {len(chunks)}")

# # Drop the last chunk if it's smaller than chunk_size
# if chunks[-1].shape[1] < chunk_size:
#     chunks = chunks[:-1]

# print(f"Total chunks after dropping last: {len(chunks)}")

tensor([  7738,    315,   3984,  ...,     23,     13, 151645])
Total chunks: 109
Total chunks after dropping last: 108


In [ ]:
# # Measure perplexity on the held-out set using the Qwen model
# import math

# total_loss = 0.0
# total_tokens = 0

# qwen_model.to(device)
# qwen_model.eval()

# for i, chunk in enumerate(chunks):
#     with torch.no_grad():
#         outputs = qwen_model(chunk.to(device), labels=chunk.to(device))
#         loss = outputs.loss.item()
#         total_loss += loss * chunk.shape[1]  # Multiply by number of tokens in the chunk
#         total_tokens += chunk.shape[1]

# total_characters = sum(len(row["text"]) for _, row in df.iterrows())
# print(f"Total loss: {total_loss:.4f}")
# print(f"Total tokens: {total_tokens}")
# print(f"Total characters in held-out set: {total_characters}")
# perplexity = math.exp(total_loss / total_tokens)
# print(f"Perplexity on held-out set: {perplexity:.2f}")
# model_agnostic_perplexity = math.exp(total_loss / total_characters)
# print(f"Model-agnostic perplexity on held-out set: {model_agnostic_perplexity:.2f}")


Total loss: 241628.2097
Total tokens: 110592
Total characters in held-out set: 535430
Perplexity on held-out set: 8.89
Model-agnostic perplexity on held-out set: 1.57


# CPT Training for SmolLM

In [10]:
# Prepare training data for SmolLM2-360M model for CPT 
train_data = r"/home/aamer/smol_llm/data/stage00/cpt_train.jsonl"
train_df = pd.read_json(train_data, lines=True)
train_df = train_df['text']

# Prepare the training corpus by appending the EOS token to each text entry and joining them into a single string'
inputs = []
for i, text in enumerate(train_df):
    train_df[i] = text + tokenizer.eos_token
    tok_batch = tokenizer(train_df[i], return_tensors="pt")
    inputs.append(tok_batch["input_ids"][0])


In [11]:
# Concatenate all tokenized inputs into a single tensor
train_corpus = torch.cat(inputs, dim=0)

# Reshape 
n = (len(train_corpus) // 1024) * 1024
train_corpus = train_corpus[:n].view(-1, 1024)  # Reshape to (num_chunks, chunk_size)

In [12]:
print(next(model.parameters()).dtype)
print(next(model.parameters()).device, flush=True)

torch.float32
cuda:0


In [ ]:
# Training Loop 
lr = 2e-5 
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

warmup_steps = 250
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, 
    start_factor=0.1, 
    total_iters=warmup_steps)

model.train()
epochs = 2
batch_size = 2

history = []
global_step = 0

for epoch in range(epochs):
    # shuffle the training corpus at the beginning of each epoch
    perm = torch.randperm(train_corpus.size(0))
    train_corpus = train_corpus[perm]
    optimizer.zero_grad()
    for i in range(0, len(train_corpus), batch_size):
        batch = train_corpus[i:i + batch_size].to(device)
        outputs = model(batch, labels=batch)
        loss_value = outputs.loss
        loss_value.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        optimizer.zero_grad()
        warmup_scheduler.step()
        history.append({
            "step": global_step,
            "epoch": epoch + 1,
            "loss": loss_value.item(),
            "lr": optimizer.param_groups[0]["lr"],
        })
        global_step += 1
        
        print(f"Epoch {epoch + 1}/{epochs}, Batch {i // batch_size + 1}/{len(train_corpus) // batch_size}, Loss: {loss_value.item():.4f}", flush=True)

    # save model after epoch
    model_save_path = f"/home/aamer/smol_llm/models/smol_cpt_epoch_{epoch + 1}.pt"
    torch.save(model.state_dict(), model_save_path)
    print(f"Saved model after epoch {epoch + 1} to {model_save_path}")



Epoch 1/2, Batch 1/2715, Loss: 2.2106
Epoch 1/2, Batch 2/2715, Loss: 2.2037
Epoch 1/2, Batch 3/2715, Loss: 2.2359
Epoch 1/2, Batch 4/2715, Loss: 2.2191
Epoch 1/2, Batch 5/2715, Loss: 2.6064
Epoch 1/2, Batch 6/2715, Loss: 2.2757
Epoch 1/2, Batch 7/2715, Loss: 2.3705
Epoch 1/2, Batch 8/2715, Loss: 1.7264
Epoch 1/2, Batch 9/2715, Loss: 2.3111
Epoch 1/2, Batch 10/2715, Loss: 2.3955
Epoch 1/2, Batch 11/2715, Loss: 2.1753
Epoch 1/2, Batch 12/2715, Loss: 2.2841
Epoch 1/2, Batch 13/2715, Loss: 1.8466
Epoch 1/2, Batch 14/2715, Loss: 2.1907
Epoch 1/2, Batch 15/2715, Loss: 2.5647
Epoch 1/2, Batch 16/2715, Loss: 2.1643
Epoch 1/2, Batch 17/2715, Loss: 2.6871
Epoch 1/2, Batch 18/2715, Loss: 2.5985
Epoch 1/2, Batch 19/2715, Loss: 2.7277
Epoch 1/2, Batch 20/2715, Loss: 2.5693
Epoch 1/2, Batch 21/2715, Loss: 2.4734
Epoch 1/2, Batch 22/2715, Loss: 2.3738
Epoch 1/2, Batch 23/2715, Loss: 2.5891
Epoch 1/2, Batch 24/2715, Loss: 2.3884
Epoch 1/2, Batch 25/2715, Loss: 2.2862
Epoch 1/2, Batch 26/2715, Loss: 2.

# POST-CPT results

In [1]:
import math, torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

model_id = "HuggingFaceTB/SmolLM2-360M"
device   = "cuda"
BLOCK    = 1024
tokenizer = AutoTokenizer.from_pretrained(model_id)

# ---------- corpora ----------
held = pd.read_json("/home/aamer/smol_llm/data/stage00/cpt_heldout.jsonl", lines=True)
domain_ids = []
for t in held["text"]:
    domain_ids.extend(tokenizer(t, add_special_tokens=False)["input_ids"])
    domain_ids.append(tokenizer.eos_token_id)

wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
general_text = "".join(t for t in wiki["text"]
                       if t.strip() and not t.strip().startswith("="))
general_ids = tokenizer(general_text, add_special_tokens=False)["input_ids"]

def to_chunks(ids):
    n = (len(ids) // BLOCK) * BLOCK
    return torch.tensor(ids[:n]).view(-1, BLOCK), n

domain_chunks,  domain_n  = to_chunks(domain_ids)
general_chunks, general_n = to_chunks(general_ids)
domain_chars  = sum(len(t) for t in held["text"])
general_chars = len(general_text)

# ---------- scorer ----------
@torch.no_grad()
def ppl(model, chunks, n_chars):
    model.eval()
    total_nll, total_tok = 0.0, 0
    for i in range(0, len(chunks), 2):
        b = chunks[i:i+2].to(device)
        out = model(b, labels=b)
        ntok = b.shape[0] * (b.shape[1] - 1)
        total_nll += out.loss.item() * ntok
        total_tok += ntok
    return math.exp(total_nll / total_tok), math.exp(total_nll / n_chars)

# ---------- run both checkpoints ----------
rows = []
for tag, ckpt in [("base", None),
                  ("cpt_epoch1", "/home/aamer/smol_llm/models/smol_cpt_epoch_1.pt")]:
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32)
    if ckpt:
        model.load_state_dict(torch.load(ckpt, map_location="cpu"))
    model.to(device)

    d_tok, d_chr = ppl(model, domain_chunks,  domain_chars)
    g_tok, g_chr = ppl(model, general_chunks, general_chars)
    rows.append({"model": tag, "domain_ppl": d_tok, "domain_ppl_char": d_chr,
                 "general_ppl": g_tok, "general_ppl_char": g_chr})
    print(rows[-1], flush=True)

    del model; torch.cuda.empty_cache()

results = pd.DataFrame(rows).set_index("model")
print(results.round(4))
print("\ndomain change: {:+.2%}   general change: {:+.2%}".format(
    results.domain_ppl.iloc[1]/results.domain_ppl.iloc[0] - 1,
    results.general_ppl.iloc[1]/results.general_ppl.iloc[0] - 1))

/home/aamer/smol_llm/.venv-sft/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Token indices sequence length is longer than the specified maximum sequence length for this model (294970 > 8192). Running this sequence through the model will result in indexing errors
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 747.68it/s]


{'model': 'base', 'domain_ppl': 10.505702640501687, 'domain_ppl_char': 1.6393451119936382, 'general_ppl': 13.11813306867026, 'general_ppl_char': 1.8217303805137746}


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1147.90it/s]


{'model': 'cpt_epoch1', 'domain_ppl': 8.827022971943782, 'domain_ppl_char': 1.5804452121857904, 'general_ppl': 14.346935943302572, 'general_ppl_char': 1.8601394119071735}
            domain_ppl  domain_ppl_char  general_ppl  general_ppl_char
model                                                                 
base           10.5057           1.6393      13.1181            1.8217
cpt_epoch1      8.8270           1.5804      14.3469            1.8601

domain change: -15.98%   general change: +9.37%
